# Creating RAG Chatbots with LangChain
In this project, we’ll develop a complete AI chatbot from scratch. Using LangChain, OpenAI, and the Pinecone vector database, we’ll build a chatbot that leverages Retrieval-Augmented Generation (RAG) to provide answers informed by external knowledge.

Our chatbot will utilize a dataset from the Deepseek R1 ArXiv paper, enabling it to respond to questions about cutting-edge developments in AI.

By the end, you’ll have a fully functional chatbot and RAG pipeline capable of engaging in conversations and delivering accurate, knowledge-based responses.

# Prerequisites

Before building the chatbot, we need to install a few essential Python libraries. Here’s a quick overview of their purpose:

langchain: A framework for Generative AI that allows us to connect and orchestrate multiple language models and components to create complex chatbot workflows.

openai: The official Python client for OpenAI, which we’ll use to generate responses from the OpenAI API.

datasets: A library providing access to a wide variety of machine learning datasets, which will serve as the knowledge base for our chatbot.

pinecone-client: The official Pinecone client for Python, enabling us to store and query our chatbot’s knowledge base as vector embeddings efficiently.

In [3]:
import os
from langchain_openai import ChatOpenAI
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass(
    "Enter your OpenAI API key: "
)

chat = ChatOpenAI(
    openai_api_key=os.environ["OPENAI_API_KEY"],
    model='gpt-4o-mini'
)

System: You are a helpful assistant.

User: Hi AI, how are you today?

Assistant: I'm doing well, thank you! How can I assist you?

User: I want to learn about string theory.

Assistant:

In [4]:
from langchain.schema import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi AI, how are you today?"),
    AIMessage(content="I'm great thank you. How can I help you?"),
    HumanMessage(content="I'd like to understand string theory.")
]

The format is very similar, we're just swapped the role of "user" for HumanMessage, and the role of "assistant" for AIMessage.

We generate the next response from the AI by passing these messages to the ChatOpenAI object.

In [ ]:
res = chat(messages)
res

In [ ]:
print(res.content)

In [ ]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="Why do physicists believe it can produce a 'unified theory'?"
)
# add to messages
messages.append(prompt)

# send to chat-gpt
res = chat(messages)

print(res.content)

# Handling Hallucinations
Even though our chatbot is up and running, LLMs have inherent limitations in their knowledge. This is because they acquire all their information during the training phase, effectively compressing everything they “see” in the data into their internal parameters. This stored information is often referred to as the model’s parametric knowledge.

By default, LLMs cannot access real-time or external information outside of their training data.

This limitation becomes obvious when we ask about recent developments, such as details from the Deepseek R1 paper, where the model may produce incomplete or inaccurate responses.

In [ ]:
# add latest AI response to messages
messages.append(res)

# now create a new user prompt
prompt = HumanMessage(
    content="What is so special about Deepseek R1?"
)
# add to messages
messages.append(prompt)

# send to OpenAI
res = chat(messages)

In [ ]:
print(res.content)

At this point, our chatbot cannot provide the answer because the necessary information isn’t part of its knowledge base. Sometimes the model clearly indicates it doesn’t know, but other times it may generate an answer that seems confident even though it’s incorrect, which can be difficult to spot.

An alternative approach is to supply the LLM with external information directly through the prompt, known as source knowledge. This allows the model to use specific data provided at runtime. For example, we can try this with the Deepseek question by including the abstract from the Deepseek R1 paper in the prompt.

In [ ]:
source_knowledge = (
    "We introduce our first-generation reasoning models, DeepSeek-R1-Zero and "
    "DeepSeek-R1. DeepSeek-R1-Zero, a model trained via large-scale "
    "reinforcement learning (RL) without supervised fine-tuning (SFT) as a "
    "preliminary step, demonstrates remarkable reasoning capabilities. Through "
    "RL, DeepSeek-R1-Zero naturally emerges with numerous powerful and "
    "intriguing reasoning behaviors. However, it encounters challenges such as "
    "poor readability, and language mixing. To address these issues and "
    "further enhance reasoning performance, we introduce DeepSeek-R1, which "
    "incorporates multi-stage training and cold-start data before RL. "
    "DeepSeek-R1 achieves performance comparable to OpenAI-o1-1217 on "
    "reasoning tasks. To support the research community, we open-source "
    "DeepSeek-R1-Zero, DeepSeek-R1, and six dense models (1.5B, 7B, 8B, 14B, "
    "32B, 70B) distilled from DeepSeek-R1 based on Qwen and Llama."
)

We can feed this additional knowledge into our prompt with some instructions telling the LLM how we'd like it to use this information alongside our original query.

In [ ]:
query = "What is so special about Deepseek R1?"

augmented_prompt = f"""Using the contexts below, answer the query.

Contexts:
{source_knowledge}

Query: {query}"""

Now we feed this into our chatbot as we were before.

In [ ]:
# create a new user prompt
prompt = HumanMessage(
    content=augmented_prompt
)
# add to messages
messages.append(prompt)

# send to OpenAI
res = chat(messages)

In [ ]:
print(res.content)

# Importing the Data
In this task, we will be importing our data. We will be using the Hugging Face Datasets library to load our data. Specifically, we will be using the "jamescalam/deepseek-r1-paper-chunked" dataset. This dataset contains the Deepseek R1 paper pre-processed into RAG-ready chunks.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "jamescalam/deepseek-r1-paper-chunked",
    split="train"
)

dataset

In [ ]:
dataset[0]

# Dataset Overview
The dataset we are using is sourced from the Deepseek R1 ArXiv papers. Each entry in the dataset represents a "chunk" of text from the R1 paper.

Because most Large Language Models (LLMs) only contain knowledge of the world as it was during training, even many of the newest LLMs cannot answer questions about Deepseek R1 — at least not without this data.

# Building the Knowledge Base
We now have a dataset that can serve as our chatbot knowledge base. Our next task is to transform that dataset into the knowledge base that our chatbot can use. To do this we must use an embedding model and vector database.

We begin by initializing our Pinecone client, this requires a free API key.

In [ ]:
from pinecone import Pinecone

# get API key at app.pinecone.io
api_key = os.getenv("PINECONE_API_KEY") or getpass(
    "Enter your Pinecone API key: "
)

# initialize client
pc = Pinecone(api_key=api_key)

In [ ]:
from pinecone import ServerlessSpec, CloudProvider, AwsRegion, Metric

index_name = "deepseek-r1-rag"

if not pc.has_index(name=index_name):
    pc.create_index(
        name=index_name,
        metric=Metric.DOTPRODUCT,
        dimension=1536,  # this aligns with text-embedding-3-small dims
        spec=ServerlessSpec(
            cloud=CloudProvider.AWS,
            region=AwsRegion.US_EAST_1
        )
    )

index = pc.Index(name=index_name)

Our index is now ready but it's empty. It is a vector index, so it needs vectors. As mentioned, to create these vector embeddings we will OpenAI's text-embedding-small-3 model — we can access it via LangChain like so:

In [ ]:
from langchain_openai import OpenAIEmbeddings

embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

res = embed_model.embed_documents(texts)
len(res), len(res[0])

From this we get two (aligning to our two chunks of text) 1536-dimensional embeddings.

We're now ready to embed and index all our our data! We do this by looping through our dataset and embedding and inserting everything in batches.

In [ ]:
from tqdm.auto import tqdm  # for progress bar

data = dataset.to_pandas()  # this makes it easier to iterate over the dataset

batch_size = 100

for i in tqdm(range(0, len(data), batch_size)):
    i_end = min(len(data), i+batch_size)
    # get batch of data
    batch = data.iloc[i:i_end]
    # generate unique ids for each chunk
    ids = [f"{x['doi']}-{x['chunk-id']}" for i, x in batch.iterrows()]
    # get text to embed
    texts = [x['chunk'] for _, x in batch.iterrows()]
    # embed text
    embeds = embed_model.embed_documents(texts)
    # get metadata to store in Pinecone
    metadata = [
        {'text': x['chunk'],
         'source': x['source']} for i, x in batch.iterrows()
    ]
    # add to Pinecone
    index.upsert(vectors=zip(ids, embeds, metadata))

In [ ]:
index.describe_index_stats()

# Retrieval Augmented Generation
We've built a fully-fledged knowledge base. Now it's time to link that knowledge base to our chatbot. To do that we'll be diving back into LangChain and reusing our template prompt from earlier.

To use LangChain here we need to load the LangChain abstraction for a vector index, called a vectorstore. We pass in our vector index to initialize the object.

In [ ]:
from langchain_pinecone import PineconeVectorStore

text_field = "text"  # the metadata field that contains our text

# initialize the vector store object
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embed_model,
    text_key=text_field
)

Using this vectorstore we can already query the index and see if we have any relevant information given our question about Llama 2.

In [ ]:
query = "What is so special about Deepseek R1?"

vectorstore.similarity_search(query, k=3)

We return a lot of text here and it's not that clear what we need or what is relevant. Fortunately, our LLM will be able to parse this information much faster than us. All we need is to link the output from our vectorstore to our chat chatbot. To do that we can use the same logic as we used earlier.

In [ ]:
def augment_prompt(query: str):
    # get top 3 results from knowledge base
    results = vectorstore.similarity_search(query, k=3)
    # get the text from the results
    source_knowledge = "\n".join([x.page_content for x in results])
    # feed into an augmented prompt
    augmented_prompt = f"""Using the contexts below, answer the query.

    Contexts:
    {source_knowledge}

    Query: {query}"""
    return augmented_prompt

In [ ]:
print(augment_prompt(query))

There is still a lot of text here, so let's pass it onto our chat model to see how it performs.

In [ ]:
# create a new user prompt
prompt = HumanMessage(
    content=augment_prompt(query)
)
# add to messages
messages.append(prompt)

res = chat(messages)

print(res.content)

We can continue with another Deepseek R1:

In [ ]:
prompt = HumanMessage(
    content=augment_prompt(
        "how does deepseek r1 compare to deepseek r1 zero?"
    )
)

res = chat(messages + [prompt])
print(res.content)

In [ ]:
pc.delete_index(index_name)